In [3]:
# Chest X-ray Dataset Cleaning Pipeline

## 1. Problem: Corrupted Images

# Corrupted images break training pipelines and cause runtime errors.

### Solution

# Detect unreadable images and remove them.

### Code

In [5]:
from PIL import Image
import os

BASE = "data/processed"
bad_files = []

for root, _, files in os.walk(BASE):
    for f in files:
        path = os.path.join(root, f)
        try:
            with Image.open(path) as img:
                img.verify()
        except Exception:
            bad_files.append(path)

print("Corrupted:", len(bad_files))
for f in bad_files[:10]:
    print(f)

# Optional removal
for f in bad_files:
    os.remove(f)

""" 
    Explanation
    img.verify() checks file integrity without loading full image
    Any failure → file is unusable → removed
"""

Corrupted: 0


' \n    Explanation\n    img.verify() checks file integrity without loading full image\n    Any failure → file is unusable → removed\n'

In [6]:
# 2. Problem: Duplicate Images

'''
    Duplicates:

        bias training
        cause data leakage
        inflate metrics
'''

##Solution
    #ash images and remove duplicates

'\n    Duplicates:\n\n        bias training\n        cause data leakage\n        inflate metrics\n'

In [8]:
import hashlib
from collections import defaultdict
import os

BASE = "data/processed"
hash_map = defaultdict(list)

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

for root, _, files in os.walk(BASE):
    for f in files:
        path = os.path.join(root, f)
        h = file_hash(path)
        hash_map[h].append(path)

duplicates = [v for v in hash_map.values() if len(v) > 1]

print("Duplicate groups:", len(duplicates))

# remove duplicates
for group in duplicates:
    for p in group[1:]:
        os.remove(p)

print("Removed duplicates.")
print("Explanation")
print("\tMD5 hash uniquely identifies file content")
print("\tdentical hash → duplicate image")
print("\tkeep one, remove others")


Duplicate groups: 0
Removed duplicates.
Explanation
	MD5 hash uniquely identifies file content
	dentical hash → duplicate image
	keep one, remove others


In [11]:
print("3. Problem: Data Leakage (Across Splits) => Same image appearing in train/val/test → invalid evaluation.")
print("Solution : Compare hashes across splits.")

def collect_hashes(split):
    hashes = set()
    for root, _, files in os.walk(f"{BASE}/{split}"):
        for f in files:
            path = os.path.join(root, f)
            hashes.add(file_hash(path))
    return hashes

train_h = collect_hashes("train")
val_h   = collect_hashes("val")
test_h  = collect_hashes("test")

print("Train ∩ Val:", len(train_h & val_h))
print("Train ∩ Test:", len(train_h & test_h))
print("Val ∩ Test:", len(val_h & test_h))

3. Problem: Data Leakage (Across Splits) => Same image appearing in train/val/test → invalid evaluation.
Solution : Compare hashes across splits.
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [ ]:
from pathlib import Path
from collections import Counter
import os

print("Problem: Class Imbalance → Imbalanced classes bias the model")
print("Solution: Count samples per class and verify balance")

# FIX: convert BASE to Path
BASE = Path("../data/processed").resolve()

def count_split(split):
    split_path = BASE / split

    if not split_path.exists():
        print(f"[ERROR] Missing: {split_path}")
        return {}

    counts = Counter()

    for cls in os.listdir(split_path):
        cls_path = split_path / cls
        if cls_path.is_dir():
            counts[cls] = len(os.listdir(cls_path))

    return counts

for split in ["train", "val", "test"]:
    print(split, count_split(split))
    max_splits = max(count_split(split).values())
    min_splits = min(count_split(split).values())
    if max_splits / min_splits > 3:
        print(f"Warning: Class imbalance in {split} (max/min > 3)")
    else:
        print(f"Data is reasonably balanced in {split} : max/min = {max_splits / min_splits:.2f}")
    

Problem: Class Imbalance → Imbalanced classes bias the model
Solution: Count samples per class and verify balance
train Counter({'normal': 560, 'pneumonia': 560, 'pneumothorax': 558, 'tuberculosis': 556, 'cardiomegaly': 552, 'effusion': 537})
Data is reasonably balanced in train
val Counter({'normal': 70, 'pneumonia': 70, 'tuberculosis': 69, 'pneumothorax': 68, 'cardiomegaly': 68, 'effusion': 67})
Data is reasonably balanced in val
test Counter({'normal': 70, 'cardiomegaly': 70, 'tuberculosis': 69, 'pneumonia': 68, 'pneumothorax': 68, 'effusion': 66})
Data is reasonably balanced in test


In [18]:
print("Problem: Mixed sizes → unstable training.")
from PIL import Image

sizes = set()

for root, _, files in os.walk(BASE):
    for f in files[:1000]:
        path = os.path.join(root, f)
        try:
            with Image.open(path) as img:
                sizes.add(img.size)
        except:
            pass

print("Unique sizes:", sizes)

Problem: Mixed sizes → unstable training.
Unique sizes: {(224, 224)}


In [19]:
print("Problem: Inconsistent Channels → Mixed formats (RGB, RGBA, grayscale) → inconsistent inputs.")
modes = set()

for root, _, files in os.walk(BASE):
    for f in files[:1000]:
        path = os.path.join(root, f)
        try:
            with Image.open(path) as img:
                modes.add(img.mode)
        except:
            pass

print("Modes:", modes)

Problem: Inconsistent Channels → Mixed formats (RGB, RGBA, grayscale) → inconsistent inputs.
Modes: {'L'}


In [20]:
print("Problem: Inconsistent Format / Size / Channels → Unstable training.")

import os
from PIL import Image

BASE = "data/processed"
TARGET_SIZE = (224, 224)

for root, _, files in os.walk(BASE):
    for f in files:
        path = os.path.join(root, f)

        try:
            img = Image.open(path)

            # remove alpha channel
            if img.mode == "RGBA":
                img = img.convert("RGB")

            # convert to grayscale
            img = img.convert("L")

            # resize
            img = img.resize(TARGET_SIZE)

            # save as PNG
            new_path = os.path.splitext(path)[0] + ".png"
            img.save(new_path)

            if new_path != path:
                os.remove(path)

        except Exception:
            os.remove(path)

print("Normalization done.")

Problem: Inconsistent Format / Size / Channels → Unstable training.
Normalization done.


In [22]:
print("""
Explanation
1. Convert RGBA → RGB (remove alpha)
2. Convert all to grayscale (L mode)
3. Resize to uniform size (224x224)
4. Save as PNG for consistency
5. Remove original if format changed
6. Remove unreadable files
""")


Explanation
1. Convert RGBA → RGB (remove alpha)
2. Convert all to grayscale (L mode)
3. Resize to uniform size (224x224)
4. Save as PNG for consistency
5. Remove original if format changed
6. Remove unreadable files



In [23]:
# Post-cleaning dataset limitations (concise, professor-style, executable cell)

from pathlib import Path
import os

BASE = Path("data/processed").resolve()

print("Post-Cleaning Dataset Limitations\n")

# 1) Structure check
required_splits = ["train", "val", "test"]
missing = [s for s in required_splits if not (BASE / s).exists()]

if missing:
    print(f"[ERROR] Missing splits: {missing}")
else:
    print("Structure: train/val/test detected")

# 2) Class consistency check
classes_per_split = {}
for split in required_splits:
    split_path = BASE / split
    if not split_path.exists():
        classes_per_split[split] = set()
        continue
    classes_per_split[split] = {
        d.name for d in split_path.iterdir() if d.is_dir()
    }

all_classes = set.union(*classes_per_split.values()) if classes_per_split else set()
inconsistent = {
    split: sorted(list(all_classes - classes_per_split.get(split, set())))
    for split in required_splits
    if classes_per_split.get(split, set()) != all_classes
}

if inconsistent:
    print(f"[WARNING] Inconsistent classes across splits: {inconsistent}")
else:
    print("Class consistency: OK")

# 3) Minimal counts sanity
def count_images(dir_path):
    try:
        return sum(
            1 for f in os.listdir(dir_path)
            if (dir_path / f).is_file()
        )
    except Exception:
        return 0

counts = {}
for split in required_splits:
    split_path = BASE / split
    counts[split] = {}
    if split_path.exists():
        for cls in classes_per_split.get(split, []):
            cls_path = split_path / cls
            counts[split][cls] = count_images(cls_path)

print("\nCounts (per split):")
for split in required_splits:
    print(split, counts.get(split, {}))

# 4) Report inherent limitations (non-structural)
print("\nLimitations:")
print("- Labels may contain noise (weak supervision in large CXR datasets).")
print("- Certain classes exhibit radiographic overlap (e.g., pneumonia vs effusion).")
print("- Multi-source data introduces distribution variability.")
print("- Patient-level independence is not guaranteed.")
print("- Class difficulty is heterogeneous; accuracy alone is insufficient.\n")

print("Status: Structurally clean dataset. Clinical limitations remain.")

Post-Cleaning Dataset Limitations

[ERROR] Missing splits: ['train', 'val', 'test']
Class consistency: OK

Counts (per split):
train {}
val {}
test {}

Limitations:
- Labels may contain noise (weak supervision in large CXR datasets).
- Certain classes exhibit radiographic overlap (e.g., pneumonia vs effusion).
- Multi-source data introduces distribution variability.
- Patient-level independence is not guaranteed.
- Class difficulty is heterogeneous; accuracy alone is insufficient.

Status: Structurally clean dataset. Clinical limitations remain.
